# 3 · Train from scratch on example simulations
Before starting, read [README.md](README.md). Run cells in order. Example files are from DECODE’s experiment, not the Nanoimager. Do not interpret a successful run as validated microscopy results. Nothing trains or writes data unless you explicitly enable it.
This prepares a **short demonstration**, not a converged network. Training is on simulated images with known positions, not supervised labels from your TIFF. The bundled example model’s camera/PSF are not yours. For your own data create a validated calibration and set camera, background, emitter and scaling parameters before training; often you need separate models per channel. Use a CUDA GPU if available; CPU training can be very slow.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile

# Change this on the lab PC. It must contain experimental_data_workflow/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'experimental_data_workflow').exists() and (ROOT.parent / 'experimental_data_workflow').exists():
    ROOT = ROOT.parent
EXAMPLE = ROOT / 'experimental_data_workflow'
WORK = ROOT / 'decode_work'
print('Project:', ROOT, '| Example exists:', EXAMPLE.exists())


In [ ]:
import torch
import decode.utils.param_io
print('CUDA:', torch.cuda.is_available())
param = decode.utils.param_io.load_params(str(EXAMPLE / 'param_run.yaml'))
param.InOut.calibration_file = str((EXAMPLE / 'spline_calibration_3dcal.mat').resolve())
param.InOut.checkpoint_init = None
param.Hardware.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
param.Hardware.device_simulation = param.Hardware.device
param.Hardware.num_worker_train = 0  # safer on Windows
param.HyperParameter.batch_size = 8
param.HyperParameter.epochs = 2  # demonstration only
param.HyperParameter.pseudo_ds_size = 128
param.TestSet.test_size = 32
print('Device:', param.Hardware.device)


### Save a parameter file
This uses a separate output directory. File paths are absolute for the PC that creates the file; recreate it if moving computers. On Windows, use a terminal for training as recommended by DECODE. Set `WRITE_CONFIG = True` only when ready.


In [ ]:
WRITE_CONFIG = False
if WRITE_CONFIG:
    WORK.mkdir(parents=True, exist_ok=True)
    (WORK / 'scratch_runs').mkdir(exist_ok=True)
    param.InOut.experiment_out = str((WORK / 'scratch_runs').resolve())
    config_path = WORK / 'scratch_params.yaml'
    if config_path.exists(): raise FileExistsError(f'Choose a fresh path: {config_path}')
    decode.utils.param_io.save_params(str(config_path), param)
    print('Config saved:', config_path)
    print('In a terminal with the DECODE environment activated, run:')
    print(f'decode.train -p "{config_path}"')
else:
    print('Set WRITE_CONFIG = True to save a new config and display the command.')


In the terminal, run the printed command. Track the generated run directory: use its `param_run.yaml` and saved `model_*.pt` together in notebook 05. Do not use this two-epoch run for scientific conclusions. If it fails, preserve the output and error.
